# MDN RAG: fine-tuning E5 на GPU

Выберите GPU в настройках Colab. Платные API не используются. Данные train — слабая синтетическая разметка; v2 проверен ассистентом, независимая человеческая проверка остаётся отдельным этапом. Страницы всех eval и резерв кандидатов исключены из train. Результаты notebook не заполнены заранее.


In [ ]:
!git clone --branch codex/rag-quality https://github.com/Pavlo-2201/rag-document-assistant.git
%cd rag-document-assistant
%pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -r requirements-training.txt
%pip install faiss-cpu==1.15.1 scikit-learn==1.9.1 snowballstemmer==3.1.1 requests==2.34.2 tqdm==4.70.1


In [ ]:
import torch
assert torch.cuda.is_available(), 'Включите GPU в настройках среды выполнения'
print(torch.__version__, torch.cuda.get_device_name())
from huggingface_hub import snapshot_download
from scripts.download_quality_models import ENCODER_FILES
snapshot_download('intfloat/multilingual-e5-small', revision='614241f622f53c4eeff9890bdc4f31cfecc418b3', allow_patterns=ENCODER_FILES, local_dir='models/e5-small')
snapshot_download('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', revision='1427fd652930e4ba29e8149678df786c240d8825', allow_patterns=['config.json','tokenizer.json','tokenizer_config.json','special_tokens_map.json','sentencepiece.bpe.model','model.safetensors'], local_dir='models/reranker')


In [ ]:
!python scripts/download_corpus.py --config configs/corpus.yaml
!python scripts/ingest.py --config configs/corpus.yaml
!python scripts/build_chunks.py --chunk-size 1000 --overlap 150
!python scripts/filter_training.py


In [ ]:
!python scripts/train_e5.py --train data/train/synthetic-v1/questions.jsonl --review data/train/synthetic-v1/auto_review.csv --exclude data/eval/qa.jsonl --exclude data/eval/qa-v2.jsonl --reserved-docs data/eval/candidates-v2-json/reserved_docs.json --batch-size 16 --epochs 2


In [ ]:
!python scripts/build_index.py --model e5-small --device cuda --precision float16
!python scripts/build_index.py --model e5-finetuned --device cuda --precision float16
!python scripts/benchmark_quality.py --phase retrieval --qa data/eval/qa-v2.jsonl --split dev_v2 --systems dense-small dense-tuned hybrid-tuned60 --output results/colab


Сначала выберите конфигурацию на dev_v2 и запишите выбор. Затем запускайте test_v2 один раз. Старый test уже использован при анализе ошибок и является диагностическим, а не новым holdout. Интервалы в retrieval_summary — парный bootstrap по одинаковым id. Скорость с кешем оценок reranker не является честным latency-бенчмарком; генераторы сравниваются отдельно без кеша ответов.


In [ ]:
import json
from pathlib import Path
print(Path('models/e5-finetuned/training_manifest.json').read_text())
print(Path('results/colab/retrieval_summary__dev_v2.json').read_text())


In [ ]:
# Небольшой архив отчётов; веса остаются отдельно в models/e5-finetuned.
import shutil
from google.colab import files
shutil.copy2('models/e5-finetuned/training_manifest.json', 'results/colab/training_manifest.json')
shutil.copy2('models/e5-finetuned/loss.jsonl', 'results/colab/loss.jsonl')
shutil.make_archive('/content/mdn-rag-reports', 'zip', 'results/colab')
files.download('/content/mdn-rag-reports.zip')
